# M?ridien Z?ro ? Serveur GPU Anime 2D (Colab T4 gratuit)

1. Dans le menu Colab : **Ex?cution > Modifier le type d'ex?cution > GPU T4**.
2. Ex?cute les 3 cellules dans l'ordre.
3. Copie l'URL `https://....trycloudflare.com` affich?e ? la cellule 3 dans ton `.env` local :
   `COLAB_API_URL=https://....trycloudflare.com`


In [ ]:
!nvidia-smi
!pip install -q --upgrade diffusers transformers accelerate safetensors fastapi uvicorn pillow
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("OK -> D?pendances et cloudflared pr?ts")


In [ ]:
import torch
from diffusers import DiffusionPipeline, EulerAncestralDiscreteScheduler

# Mod?le natif anime 2D (alternative : "cagliostrolab/animagine-xl-3.1")
MODEL_ID = "cagliostrolab/animagine-xl-4.0"

pipe = DiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
pipe.to("cuda")
pipe.enable_vae_slicing()
pipe.enable_attention_slicing()

print(f"OK -> Mod?le {MODEL_ID} charg? sur {torch.cuda.get_device_name(0)}")


In [ ]:
import io
import re
import subprocess
import threading
import time
import uvicorn
from fastapi import FastAPI, Response
from pydantic import BaseModel

app = FastAPI(title="Anime Engine Colab GPU")
gpu_lock = threading.Lock()


class GenerateRequest(BaseModel):
    prompt: str
    negative_prompt: str = (
        "lowres, bad anatomy, bad hands, text, error, missing finger, extra digits, "
        "cropped, worst quality, low quality, jpeg artifacts, watermark, 3d render, photorealistic"
    )
    width: int = 1216
    height: int = 832
    steps: int = 28
    cfg_scale: float = 6.0
    seed: int = 42


@app.get("/health")
def health():
    return {"status": "ok", "model": MODEL_ID}


@app.post("/generate")
def generate(req: GenerateRequest):
    w = max(512, min(1344, (req.width // 64) * 64))
    h = max(512, min(1344, (req.height // 64) * 64))
    generator = torch.Generator(device="cuda").manual_seed(int(req.seed))

    with gpu_lock:
        image = pipe(
            prompt=req.prompt,
            negative_prompt=req.negative_prompt,
            width=w,
            height=h,
            num_inference_steps=req.steps,
            guidance_scale=req.cfg_scale,
            generator=generator,
        ).images[0]

    buf = io.BytesIO()
    image.save(buf, format="PNG")
    return Response(content=buf.getvalue(), media_type="image/png")


def run_server():
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning")


threading.Thread(target=run_server, daemon=True).start()
time.sleep(2)

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

for line in proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        print("=" * 65)
        print("SERVEUR PRET ! Colle cette ligne dans ton .env local :")
        print(f"COLAB_API_URL={match.group(0)}")
        print("=" * 65)
        break

try:
    proc.wait()
except KeyboardInterrupt:
    proc.terminate()
    print("Serveur arr?t?.")
